# Bai existing candidate re-evaluation

Re-evaluates the first real trained Bai adapter under the corrected shared SFT/eval prompt contract. No retraining, no release creation, unchanged promotion gate. If rejected, produces failure clusters and a pending human-review queue.


In [ ]:
import os, sys, subprocess, json, shutil, zipfile
from pathlib import Path
print('Python', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51,<5', 'peft>=0.15,<1', 'datasets>=3,<5', 'accelerate', 'bitsandbytes', 'sentencepiece', 'huggingface-hub', 'kagglehub'], check=True)
import kagglehub
print('kagglehub ready')


In [ ]:
ROOT=Path('/kaggle/working/tamdeshevle')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1','https://github.com/eneonstudio-dev/tamdeshevle.git',str(ROOT)],check=True)
print('repo', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
SOURCE='eneonstii/notebook07f42bd563/versions/1'
downloaded=Path(kagglehub.notebook_output_download(SOURCE))
print('downloaded', downloaded)
zips=list(downloaded.rglob('bai-candidate-experiment.zip'))
if not zips: raise FileNotFoundError('bai-candidate-experiment.zip not found in Version 1 output')
candidate_root=Path('/kaggle/working/first-bai-candidate')
if candidate_root.exists(): shutil.rmtree(candidate_root)
candidate_root.mkdir(parents=True)
with zipfile.ZipFile(zips[0]) as z: z.extractall(candidate_root)
adapter=candidate_root/'adapter'
if not adapter.is_dir(): raise FileNotFoundError(f'adapter missing after extract: {adapter}')
print('adapter', adapter)


In [ ]:
seed=Path('/kaggle/working/bai-reeval-seed')
if seed.exists(): shutil.rmtree(seed)
subprocess.run(['node',str(ROOT/'teacher-lab/training/deterministic-seed.mjs'),str(seed)],cwd=ROOT,check=True)
eval_gold=seed/'eval-gold.jsonl'
print('heldout rows', sum(1 for line in eval_gold.open(encoding='utf-8') if line.strip()))


In [ ]:
out=Path('/kaggle/working/bai-candidate-reeval')
if out.exists(): shutil.rmtree(out)
cmd=[sys.executable,str(ROOT/'teacher-lab/training/reevaluate_candidate.py'),'--eval-gold',str(eval_gold),'--candidate-adapter',str(adapter),'--out',str(out)]
result=subprocess.run(cmd,cwd=ROOT)
print('reeval exit', result.returncode)
manifest=json.loads((out/'reeval-manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest,ensure_ascii=False,indent=2))


In [ ]:
pred=out/'candidate-predictions.jsonl'
if not pred.is_file(): raise FileNotFoundError('candidate predictions missing; re-evaluation did not reach inference output')
fail=out/'failure-analysis'
subprocess.run([sys.executable,str(ROOT/'teacher-lab/training/analyze_candidate_failures.py'),'--eval-gold',str(eval_gold),'--predictions',str(pred),'--out-dir',str(fail)],cwd=ROOT,check=True)
print((fail/'failure-summary.json').read_text(encoding='utf-8'))
print('Artifacts:')
for p in sorted(out.rglob('*')):
    if p.is_file(): print(p)
